# Backpropagation

> The chain rule applied mechanically, right to left, working out every gradient in a network for the price of one extra pass. Derived, written, and checked.

Read this chapter at `/learn/09-backpropagation/`. Exported from `src/content/chapters/09-backpropagation.mdx` — edit there, not here.


Yesterday's numerical gradient cost two forward passes **per parameter**.

Backpropagation costs about one forward pass **in total** — for every parameter
at once. Today is where that comes from.

It is the most important procedure in the field, and it is not difficult. It is
the chain rule from school calculus, applied step by step,
working backwards. If you can do the chain rule, you can do this. The difficulty
is all in the reputation.

<Says who="moss">
If the phrase "chain rule" makes your stomach drop a little: it means that when
one thing depends on a second thing, which depends on a third, you can work out
how the first responds to the third by multiplying the two steps together. That
is the entire tool. Everything today is that, done several times in a row.
</Says>

## The idea, on something tiny

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# f(x) = (3x + 1)^2, decomposed into steps
def forward_verbose(x):
    a = 3 * x        # step 1
    b = a + 1        # step 2
    c = b ** 2       # step 3
    return a, b, c

a, b, c = forward_verbose(2.0)
print(f"x=2  ->  a={a}  b={b}  c={c}")

We want $dc/dx$. So let's work *backwards*, asking the same small question at
every step: **how much does my output change when my input changes?**

In [ ]:
x = 2.0
a, b, c = forward_verbose(x)

dc_dc = 1.0            # trivially, c changes 1:1 with itself
dc_db = dc_dc * 2 * b  # c = b^2      =>  dc/db = 2b
dc_da = dc_db * 1      # b = a + 1    =>  db/da = 1
dc_dx = dc_da * 3      # a = 3x       =>  da/dx = 3

print(f"backprop : dc/dx = {dc_dx}")
print(f"analytic : dc/dx = {2 * (3 * x + 1) * 3}")
print(f"numeric  : dc/dx = {(forward_verbose(x + 1e-6)[2] - forward_verbose(x - 1e-6)[2]) / 2e-6:.6f}")

Three completely different methods, one answer. Good.

But now look at the *structure* of that backward pass, because the structure is
the whole point:

**Each line is the previous line, times one local derivative.**

Nobody had to know the formula for the whole composition. Nobody expanded
$(3x+1)^2$. Each step knew only its own derivative and the number handed to it
from the right, multiplied the two, and passed the result further back.

That's a bucket brigade. Each operation only has to know its own tiny job.

That is the whole algorithm.

A quantity flows backwards through the graph — the derivative of the loss with
respect to *this node's output* — and each operation multiplies it by its own
local derivative before passing it further back.

The flowing quantity is universally called the **gradient**, or in older texts
delta ($\delta$). PyTorch calls it `grad_output` inside an autograd function.
Same thing, three names.

## Why it's cheap

Consider a chain $x \to h_1 \to h_2 \to \dots \to h_n \to L$.

The forward pass computes each $h_i$ once: $n$ operations.

The backward pass computes each $\partial L/\partial h_i$ once, right to left,
reusing the value from the step before: another $n$ operations.

So the gradient with respect to *everything* costs about the same as evaluating
the function once. Not per parameter. Total.

Here's the part I think is lovely, and it's a fact about *bracketing*.

The chain rule doesn't specify an order. For a function
$\mathbb{R}^n \to \mathbb{R}^m$ the derivative is a Jacobian matrix, and a
composition's Jacobian is a product of Jacobians:

$$
J = J_n \cdot J_{n-1} \cdots J_2 \cdot J_1
$$

Matrix multiplication is associative — so you may bracket that product however
you like, and every bracketing gives the *same answer*.

But they don't cost the same. Not remotely.

**Forward mode** brackets left to right, propagating derivatives with respect to
one *input* forward. Cost is proportional to the number of inputs.

**Reverse mode** brackets right to left, propagating derivatives of one *output*
backward. Cost is proportional to the number of outputs.

Now count what we've got. In machine learning there are $10^{11}$ inputs (the
parameters) and exactly **one** output (the scalar loss).

So reverse mode is cheaper by a factor of $10^{11}$. And notice what's really
doing the work here: the fact that a loss is a *single number*. That isn't a
convention or a stylistic choice — it is the property that makes training
affordable at all.

If you wanted to optimise two things at once, truly and separately, you'd pay
twice. The entire economics of deep learning rests on collapsing everything you
care about into one scalar and then bracketing the product from the right.

This also tells you exactly when forward mode wins: few inputs, many outputs.
It's used for sensitivity analysis and inside some physics simulators, and JAX
exposes both (`jax.grad` is reverse, `jax.jvp` is forward).

Which means backpropagation isn't really a machine learning invention at all.
It's reverse-mode automatic differentiation — a general technique from numerical
computing — wearing a machine-learning-shaped name.

## Deriving it for our network

Yesterday's network:

$$
z_1 = XW_1 + b_1 \quad
a_1 = \mathrm{ReLU}(z_1) \quad
z_2 = a_1W_2 + b_2 \quad
\hat{y} = \sigma(z_2)
$$

with binary cross-entropy loss.

Five steps. Let's do them properly.

**Step 1 — the output layer, where a small miracle happens.**

Cross-entropy is $L = -\frac{1}{n}\sum [y\log\hat{y} + (1-y)\log(1-\hat{y})]$ and
$\hat{y} = \sigma(z_2)$.

Differentiate the loss with respect to $\hat{y}$ and you get a denominator of
$\hat{y}(1-\hat{y})$. Differentiate the sigmoid and you get a numerator of
$\hat{y}(1-\hat{y})$.

They cancel. Exactly. Everything disappears and you're left with:

$$
\frac{\partial L}{\partial z_2} = \frac{1}{n}(\hat{y} - y)
$$

Just the error. Prediction minus target, nothing else.

This is why sigmoid pairs with cross-entropy and not with squared error — the
pairing is *chosen* so the derivative comes out clean and doesn't vanish when the
model is confidently wrong. Softmax with categorical cross-entropy does precisely
the same thing for many classes.

You met this cancellation already in chapter 5, when the logistic gradient turned
out to look identical to the linear one. Now you know why.

Call this $\delta_2$.

**Step 2 — the parameters of the output layer.**

Since $z_2 = a_1W_2 + b_2$:

$$
\frac{\partial L}{\partial W_2} = a_1^{\top}\delta_2
\qquad
\frac{\partial L}{\partial b_2} = \sum_{\text{rows}} \delta_2
$$

Read the first one in words: the weight gradient is **the input to the layer,
transposed, times the error coming back**.

That pattern holds for every dense layer in every network you will ever write.
Learn it once and you can write the backward pass for anything.

The bias gradient sums over the batch because the bias was
broadcast across it — and the gradient of a broadcast
is a sum. Always. That's a rule worth keeping: broadcast forward, sum backward.

**Step 3 — push the error through the weights.**

$$
\frac{\partial L}{\partial a_1} = \delta_2 W_2^{\top}
$$

Forward, the activations went *through* $W_2$. Backward, the error goes through
$W_2^{\top}$.

The transpose isn't decoration. It's what makes the shapes line up, and it's
worth checking on paper exactly once so you never doubt it again.

**Step 4 — through the ReLU.**

ReLU's derivative is 1 where its input was positive and 0 everywhere else:

$$
\delta_1 = \frac{\partial L}{\partial a_1} \odot \mathbb{1}[z_1 > 0]
$$

where $\odot$ is elementwise multiplication.

So the gradient is simply **masked**. Units that were off contribute nothing and
learn nothing this step.

And there — right there in the algebra — is the "dying ReLU" problem you measured
yesterday. A unit that's always negative is always masked, always gets zero
gradient, and can therefore never change. It's dead, and the maths shows you
exactly why it can't come back.

**Step 5 — the input layer, identical in form to step 2.**

$$
\frac{\partial L}{\partial W_1} = X^{\top}\delta_1
\qquad
\frac{\partial L}{\partial b_1} = \sum_{\text{rows}} \delta_1
$$

Five steps — and the middle three *repeat per layer*.

A hundred-layer network is these same three lines in a loop. That's not a
simplification for teaching. That is literally what the code says.

## The implementation

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=800, noise=0.22, random_state=0)
X = (X - X.mean(0)) / X.std(0)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

def init(n_in, n_hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)), "b1": np.zeros(n_hidden),
            "W2": rng.normal(0, np.sqrt(2 / n_hidden), (n_hidden, n_out)), "b2": np.zeros(n_out)}

def forward(p, X):
    z1 = X @ p["W1"] + p["b1"]
    a1 = np.maximum(0, z1)
    z2 = a1 @ p["W2"] + p["b2"]
    y_hat = 1 / (1 + np.exp(-z2))
    return y_hat, {"z1": z1, "a1": a1, "z2": z2}   # cache what backward needs

def backward(p, cache, X, y, y_hat):
    n = len(y)
    d2 = (y_hat - y.reshape(-1, 1)) / n            # step 1  (n, 1)
    gW2 = cache["a1"].T @ d2                       # step 2  (h, 1)
    gb2 = d2.sum(0)
    da1 = d2 @ p["W2"].T                           # step 3  (n, h)
    d1  = da1 * (cache["z1"] > 0)                  # step 4  (n, h)
    gW1 = X.T @ d1                                 # step 5  (2, h)
    gb1 = d1.sum(0)
    return {"W1": gW1, "b1": gb1, "W2": gW2, "b2": gb2}

p = init(2, 16, 1)
y_hat, cache = forward(p, X_tr)
grads = backward(p, cache, X_tr, y_tr, y_hat)
{k: v.shape for k, v in grads.items()}

Seven lines in `backward`, numbered to match the five steps you just derived.
Read them side by side — that's the whole payoff of doing the derivation.

Notice every gradient shape matches its parameter's shape. **That's the first
thing to check, always.** If `gW1.shape != p["W1"].shape`, something is
transposed, and you've found your bug before you've even run anything.

Notice also that `forward` returns a **cache**. The backward pass needs values
the forward pass computed — that's not an implementation detail, it's inherent.
It's also why training uses far more memory than inference: you have to keep
every intermediate activation around until the gradient comes back for it.

## Gradient checking

Never trust a hand-written backward pass. Not yours, not mine. Verify it against
the definition.

In [ ]:
def loss_of(p, X, y):
    y_hat, _ = forward(p, X)
    y_hat = np.clip(y_hat.ravel(), 1e-9, 1 - 1e-9)
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)).mean()

def numerical(p, X, y, key, idx, eps=1e-6):
    orig = p[key][idx]
    p[key][idx] = orig + eps; hi = loss_of(p, X, y)
    p[key][idx] = orig - eps; lo = loss_of(p, X, y)
    p[key][idx] = orig
    return (hi - lo) / (2 * eps)

Xs, ys = X_tr[:50], y_tr[:50]
y_hat, cache = forward(p, Xs)
analytic = backward(p, cache, Xs, ys, y_hat)

print(f"{'param':6s} {'index':8s} {'backprop':>12s} {'numeric':>12s} {'rel err':>10s}")
rng = np.random.default_rng(0)
worst = 0.0
for key in ["W1", "b1", "W2", "b2"]:
    for _ in range(3):
        idx = tuple(int(rng.integers(0, s)) for s in p[key].shape)
        an, nu = analytic[key][idx], numerical(p, Xs, ys, key, idx)
        rel = abs(an - nu) / max(abs(an) + abs(nu), 1e-12)
        worst = max(worst, rel)
        print(f"{key:6s} {str(idx):8s} {an:12.8f} {nu:12.8f} {rel:10.2e}")
print(f"\nworst relative error: {worst:.2e}   {'PASS' if worst < 1e-5 else 'FAIL'}")

Relative errors around $10^{-9}$ mean the derivation is right — that's floating
point noise and nothing else.

Anything above $10^{-4}$ means a bug, and the usual culprits are boringly
consistent: a missing transpose, a missing `/n`, or summing the bias gradient
over the wrong axis.

Gradient checking is how *every* autodiff library is tested, and it's what you
should do the moment you write a custom layer.

It's slow, so you run it once on fifty examples — never inside the training loop.
Think of it as a unit test for calculus.

## Now train it properly

In [ ]:
def fit(hidden=32, steps=4000, lr=0.5, seed=0):
    p = init(2, hidden, 1, seed)
    hist = []
    for _ in range(steps):
        y_hat, cache = forward(p, X_tr)
        g = backward(p, cache, X_tr, y_tr, y_hat)
        for k in p:
            p[k] -= lr * g[k]
        hist.append(loss_of(p, X_tr, y_tr))
    return p, np.array(hist)

import time
t = time.perf_counter()
p, hist = fit()
dt = time.perf_counter() - t

acc = ((forward(p, X_va)[0].ravel() > 0.5).astype(int) == y_va).mean()
print(f"4000 steps in {dt:.2f}s   final loss {hist[-1]:.4f}   validation accuracy {acc:.3f}")

Four thousand steps, in about a second, on a browser.

Now the comparison I promised you yesterday. Same network, same data, one
gradient computed both ways:

In [ ]:
def numerical_all(p, X, y, eps=1e-5):
    grads = {}
    for key, mat in p.items():
        g = np.zeros_like(mat)
        for idx in np.ndindex(mat.shape):
            o = mat[idx]
            mat[idx] = o + eps; hi = loss_of(p, X, y)
            mat[idx] = o - eps; lo = loss_of(p, X, y)
            mat[idx] = o
            g[idx] = (hi - lo) / (2 * eps)
        grads[key] = g
    return grads

q = init(2, 32, 1)
n_params = sum(v.size for v in q.values())

t = time.perf_counter()
yh, ca = forward(q, X_tr); backward(q, ca, X_tr, y_tr, yh)
back_ms = (time.perf_counter() - t) * 1000

t = time.perf_counter()
numerical_all(q, X_tr, y_tr)
num_ms = (time.perf_counter() - t) * 1000

print(f"{n_params} parameters, {len(X_tr)} examples")
print(f"  backpropagation : {back_ms:8.2f} ms")
print(f"  numerical       : {num_ms:8.2f} ms   ({num_ms / back_ms:.0f}x slower)")

And this is what makes it matter: **that ratio grows linearly with the
parameter count.**

At 129 parameters the gap is already big (I get somewhere around 80x; you'll
get your own number, since it depends on your machine and which runtime you're
on). At ten million parameters it would be a factor of millions. At a hundred
billion, the numerical version doesn't finish before the sun does.

That's the difference one algorithm makes. And it's why this is the one algorithm
worth deriving by hand, once, yourself — which you now have.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2.2, 2.4, 240), np.linspace(-2.4, 2.4, 240))
zz = forward(p, np.c_[xx.ravel(), yy.ravel()])[0].reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(8.8, 3.2))
ax[0].plot(hist); ax[0].set_yscale("log"); ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[1].contourf(xx, yy, zz, levels=24, cmap="coolwarm", alpha=.75)
ax[1].contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=1)
ax[1].scatter(X_va[:, 0], X_va[:, 1], c=y_va, s=8, cmap="coolwarm", edgecolors="none")
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].set_title(f"validation accuracy {acc:.3f}")
plt.tight_layout()

There's your crescent boundary, learned by a network you wrote every line of.

## Where gradients go wrong

**Vanishing.** Each layer multiplies the flowing gradient by its local
derivative. Sigmoid's derivative maxes at 0.25, so ten sigmoid layers scale the
gradient by at most $10^{-6}$ — and the early layers simply stop learning.

This is the reason deep networks didn't work before about 2010, and
ReLU is the largest single part of the fix, alongside residual
connections and normalisation layers.

**Exploding.** The mirror image: local derivatives above 1 compound upward until
the update is enormous and the loss becomes `nan`. The standard defence is
gradient clipping — if the gradient's norm exceeds a threshold,
scale the whole vector down and carry on.

Let's just measure both:

In [ ]:
for name, act, dact in [("sigmoid", lambda z: 1/(1+np.exp(-z)), lambda a: a*(1-a)),
                        ("relu",    lambda z: np.maximum(0,z),  lambda a: (a>0).astype(float))]:
    rng = np.random.default_rng(0)
    a = rng.normal(size=(256, 64))
    Ws = [rng.normal(0, np.sqrt(2/64), (64, 64)) for _ in range(30)]
    caches = []
    for W in Ws:
        a = act(a @ W); caches.append(a)
    g = np.ones_like(a)
    norms = []
    for W, c in zip(reversed(Ws), reversed(caches)):
        g = (g * dact(c)) @ W.T
        norms.append(np.abs(g).mean())
    print(f"{name:8s} gradient magnitude  layer 30 -> {norms[0]:.3e}   layer 1 -> {norms[-1]:.3e}"
          f"   ratio {norms[-1]/norms[0]:.2e}")

Thirty layers of sigmoid shrinks the gradient by many orders of magnitude. ReLU
holds it roughly steady.

That one measurement is the practical content of about a decade of research, and
you just ran it in a browser tab.

**Silently accumulating.** In PyTorch, `.grad` *adds* rather than overwrites — so
a missing `zero_grad()` means every step uses the sum of
every gradient so far.

Training diverges. Nothing errors. Nothing warns you. It's the most common
PyTorch bug there is, and now you'll recognise it on sight tomorrow.

**"Why does `forward` return a cache? That feels like a hack."** It isn't — the
backward pass really does need values the forward pass computed (`a1` for step 2,
`z1` for step 4). Every framework does this. It's also why training uses far more
memory than inference.

**"I can't see why `d1 = da1 * (z1 > 0)` is the ReLU derivative."** `(z1 > 0)` is
a boolean array, which multiplies as 1s and 0s. So the gradient survives where
the unit was on and is zeroed where it was off. That *is* the derivative of
`max(0, z)`, written as a mask.

**"My gradient check fails on the biases but passes on the weights."** Almost
always a `sum(0)` versus `sum(1)` mix-up. The bias was broadcast across the
batch, so the gradient sums across the batch — which is axis 0 in `(n, hidden)`.

**"Where did the `/ n` come from?"** From the `.mean()` in the loss. A mean is a
sum divided by n, and that constant carries all the way through the backward
pass. Exercise 1 below deletes it deliberately, and the result is instructive.

**"This all works for two layers. What about fifty?"** Steps 3 and 4 in a loop.
That's it — the exercise at the bottom has you add a third layer, and
once you've done that you've done all of them.

In [ ]:
# 1. Introduce a bug on purpose: delete the `/ n` in backward's `d2`.
#    Does gradient checking catch it? By what factor is it wrong?
#
# 2. Replace ReLU with tanh in forward and backward.
#    (d/dz tanh(z) = 1 - tanh(z)^2.) Does it still train? Compare loss curves.
#
# 3. Add a second hidden layer. Derive its gradients by pattern-matching
#    steps 2-4, gradient-check them, then train.

print("replace me")

Number 3 is the one that will teach you the most. Before writing any code, write
out steps 2, 3 and 4 for the new layer on paper. You'll find they're the same
three expressions with different subscripts — and *noticing that* is the whole
lesson.

In [ ]:
# 1. The missing /n
def backward_buggy(p, cache, X, y, y_hat):
    d2 = (y_hat - y.reshape(-1, 1))                # <- no / n
    gW2 = cache["a1"].T @ d2; gb2 = d2.sum(0)
    d1 = (d2 @ p["W2"].T) * (cache["z1"] > 0)
    return {"W1": X.T @ d1, "b1": d1.sum(0), "W2": gW2, "b2": gb2}

y_hat, cache = forward(p, Xs)
good = backward(p, cache, Xs, ys, y_hat)["W1"]
bad  = backward_buggy(p, cache, Xs, ys, y_hat)["W1"]
print(f"ratio bad/good = {(bad / good).mean():.1f}   (= n = {len(ys)})")

Gradient checking catches it instantly — every gradient is off by exactly the
batch size.

And here is the part that catches people out, which is worth dwelling on: **it still
trains.**

A constant factor on the gradient is indistinguishable from a larger learning
rate. So the bug works fine at batch size 50, and then quietly diverges at batch
size 512 — because now the same bug multiplies by ten times more.

That is a horrible bug to find without a gradient check. Six months later, in
production, when somebody increases the batch size for speed. Gradient checking
takes thirty seconds.

In [ ]:
# 2. tanh
def fit_tanh(hidden=32, steps=4000, lr=0.5, seed=0):
    p = init(2, hidden, 1, seed); hist = []
    for _ in range(steps):
        z1 = X_tr @ p["W1"] + p["b1"]; a1 = np.tanh(z1)
        z2 = a1 @ p["W2"] + p["b2"];   yh = 1 / (1 + np.exp(-z2))
        d2 = (yh - y_tr.reshape(-1, 1)) / len(y_tr)
        d1 = (d2 @ p["W2"].T) * (1 - a1 ** 2)
        p["W2"] -= lr * a1.T @ d2; p["b2"] -= lr * d2.sum(0)
        p["W1"] -= lr * X_tr.T @ d1; p["b1"] -= lr * d1.sum(0)
        yc = np.clip(yh.ravel(), 1e-9, 1-1e-9)
        hist.append(-(y_tr*np.log(yc) + (1-y_tr)*np.log(1-yc)).mean())
    return p, np.array(hist)

p_t, hist_t = fit_tanh()
plt.figure(figsize=(5, 3))
plt.plot(hist, label="relu"); plt.plot(hist_t, label="tanh")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend()
plt.tight_layout()
print(f"relu final {hist[-1]:.4f}   tanh final {hist_t[-1]:.4f}")

Both work perfectly well — and that's worth understanding rather than being
surprised by. The vanishing-gradient argument is about **depth**, and two layers
is not deep. Nothing has had a chance to vanish.

tanh is often slightly smoother early on, because it's zero-centred. The
difference becomes decisive somewhere past ten layers, which is exactly what the
measurement cell above was showing you.

So: don't over-apply a rule outside the range where it bites. "ReLU beats tanh" is
true at depth and roughly irrelevant here.

**Question 3** is the one worth doing.

The pattern to notice is that steps 2, 3 and 4 are *identical for every layer*:
input transposed times incoming error; error pushed back through the transposed
weights; mask by the activation derivative.

Once you've written that as a loop, you have written the core of a deep learning
framework. Which is, more or less exactly, what PyTorch is — plus GPU kernels,
plus a great deal of very good engineering.

That's tomorrow.

Tomorrow: let a library do all of this, and see how little is left of your code.